# TF-IDF + Logistic Regression — Seniority (Baseline)


In [2]:
from pathlib import Path
import pandas as pd

In [5]:
PROJECT_ROOT = Path.cwd().parents[1]
DATA_PATH = PROJECT_ROOT / "files"

df = pd.read_csv(DATA_PATH / "seniority-v2.csv")

df.shape, df.columns


((9428, 2), Index(['text', 'label'], dtype='object'))

In [6]:
df.head(5)

,text,label
0,Analyst,Junior
1,Analyste financier,Junior
2,Anwendungstechnischer Mitarbeiter,Junior
3,Application Engineer,Senior
4,Applications Engineer,Senior


In [7]:
df.isna().sum().sort_values(ascending=False).head(20)

text     0
label    0
dtype: int64

In [8]:
df["label"].value_counts()

label
Senior        3733
Lead          3546
Director       984
Management     756
Junior         409
Name: count, dtype: int64

In [9]:
from sklearn.model_selection import train_test_split

X = df["text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape

((6599,), (2829,))

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),   # Unigrams + Bigrams
            min_df=2,             # Wörter müssen mind. 2x vorkommen
            max_df=0.9,           # sehr häufige Wörter ignorieren
            lowercase=True
        )
    ),
    (
        "clf",
        LogisticRegression(
            max_iter=1000,
            n_jobs=-1,
            class_weight="balanced"  # wichtig wegen Lead/Director
        )
    )
])

In [11]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.9, min_df=2, ngram_range=(1, 2))),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    n_jobs=-1))])

In [12]:
y_pred = pipe.predict(X_test)

In [13]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    Director       0.99      0.97      0.98       295
      Junior       0.88      1.00      0.94       123
        Lead       0.97      0.98      0.97      1064
  Management       0.93      0.94      0.93       227
      Senior       0.99      0.96      0.98      1120

    accuracy                           0.97      2829
   macro avg       0.95      0.97      0.96      2829
weighted avg       0.97      0.97      0.97      2829



In [14]:
import pandas as pd

cm = pd.crosstab(
    y_test,
    y_pred,
    rownames=["truth"],
    colnames=["pred"],
    normalize="index"
).round(3)

cm

pred,Director,Junior,Lead,Management,Senior
truth,,,,,
Director,0.969,0.000,0.024,0.000,0.007
Junior,0.000,1.000,0.000,0.000,0.000
Lead,0.000,0.005,0.982,0.006,0.008
Management,0.009,0.009,0.040,0.938,0.004
Senior,0.001,0.009,0.019,0.009,0.962


In [15]:
import numpy as np

tfidf = pipe.named_steps["tfidf"]
clf = pipe.named_steps["clf"]

feature_names = np.array(tfidf.get_feature_names_out())
classes = clf.classes_

top_n = 15

for i, cls in enumerate(classes):
    top_features = feature_names[np.argsort(clf.coef_[i])[-top_n:]]
    print(f"\nTop features for class '{cls}':")
    print(", ".join(top_features))


Top features for class 'Director':
senior director, director head, director business, managing directors, abteilungsdirektor, verkaufsdirektor, marketing director, managing, vertriebsdirektor, director marketing, director of, directors, sales director, director sales, director

Top features for class 'Junior':
junior manager, assistentin marketing, crm, mitarbeiterin marketing, associated, marketing, associate, assistent, assistentin, mitarbeiter, referent, mitarbeiterin, referentin, analyst, junior

Top features for class 'Lead':
verkaufsleiter, teamleiterin, teamleitung, of, bereichsleiter, abteilungsleiter, projektleiter, geschäftsleitung, teamleiter, leiterin, head of, head, vertriebsleiter, leitung, leiter

Top features for class 'Management':
der geschäftsführung, product owner, geschäftsführerin, officer, president, vice president, cio, vice, founder, chief, owner, geschäftsführung, ceo, vp, geschäftsführer

Top features for class 'Senior':
account, senior vice, assistante, dev

Apply trained TF-IDF Seniority Model to LinkedIn CVs

In [17]:
import json

with open(DATA_PATH / "linkedin-cvs-annotated.json", "r", encoding="utf-8") as f:
    cvs = json.load(f)


In [18]:
jobs = []

for person_id, cv in enumerate(cvs):
    for job in cv:
        if job["status"] == "ACTIVE":
            jobs.append({**job, "person_id": person_id})

df_active = pd.DataFrame(jobs)
df_active.shape

(623, 9)

In [19]:
df_active["text"] = df_active["position"].fillna("").str.lower()

In [20]:
df_active["seniority_pred_tfidf"] = pipe.predict(df_active["text"])

In [21]:
pd.crosstab(
    df_active["seniority"],
    df_active["seniority_pred_tfidf"],
    normalize="index"
).round(3)

seniority_pred_tfidf,Director,Junior,Lead,Management,Senior
seniority,,,,,
Director,0.853,0.000,0.147,0.000,0.000
Junior,0.000,0.333,0.417,0.000,0.250
Lead,0.000,0.008,0.720,0.032,0.240
Management,0.109,0.000,0.182,0.589,0.120
Professional,0.000,0.032,0.625,0.019,0.324
Senior,0.023,0.045,0.159,0.000,0.773


In [22]:
df_active["seniority"]            # true label
df_active["seniority_pred_tfidf"] # prediction

0            Lead
1            Lead
2            Lead
3            Lead
4            Lead
          ...    
618          Lead
619          Lead
620    Management
621          Lead
622          Lead
Name: seniority_pred_tfidf, Length: 623, dtype: object

In [23]:
from sklearn.metrics import accuracy_score

mask = df_active["seniority"].notna()

accuracy = accuracy_score(
    df_active.loc[mask, "seniority"],
    df_active.loc[mask, "seniority_pred_tfidf"]
)

accuracy

0.4333868378812199